In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

2.7.0+cu128
True
12.8
NVIDIA GeForce RTX 3080 Ti


`transformers` s a Python library from Hugging Face for working with pretrained transformer models.
It helps you load and use models for tasks like:

* pretrained language models
* tokenizers
* embeddings
* text generation
* translation
* summarization
* question answering

In [ ]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

import os
import transformers.models.t5
print(transformers.models.t5.__file__)

`transformers.models.t5` is the T5 model module inside the Hugging Face transformers library.

It is not a model instance by itself. It is a Python package/module that contains the code for T5-related classes, such as:

model classes
configuration classes
tokenizer support
T5 architecture implementation

In [ ]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("t5-small")
num_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {num_params:,}")

total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Model size: {total_bytes / 1024**2:.2f} MB")

In [ ]:
# Load the model and tokenizer locally
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# model_name = "google/flan-t5-large"  # You can also use "google/flan-t5-xl"
model_name = 'google/flan-t5-large'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

这里不用T5 因为版本有问题
* model='Qwen/Qwen2.5-0.5B-Instruct' 垃圾
* model='microsoft/phi-2' 也不行


In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

pipe = pipeline(
    "text-generation",
    model='microsoft/phi-2',
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=100, 
)


In [ ]:
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

#if not already imported earlier
from langchain_core.prompts import PromptTemplate
# Define prompt template
template = """Question: {question}
Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

#### Option 1(Older version) `langchain==1.2.15` not support

In [ ]:
# from langchain.chains import LLMChain
# # Create LLMChain
# llm_chain = LLMChain(prompt=prompt, llm=llm)

# # Run model for each question
# for question in questions:
#     response = llm_chain.run(question)
#     print(f"Q: {question}\nA: {response}\n")

#### Option 2(newer version) 

In [ ]:
from langchain_core.runnables import RunnableSequence
# # RunnableSequence
chain = prompt | llm

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))